In [2]:
# Outcome Model 05 v1: Random forest, no winsorization
# Hyperparameters tuned by six-fold expanding-window CV through 2021
# Thesis-standard implementation under final outcome horse-race plan v1

import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Outcome_Master_Modeling_Panel_v1.csv"

TUNING_SUMMARY_CSV = BASE_DIR / "Outcome_Model_05_RF_NoWinsor_v1_Tuning_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Outcome_Model_05_RF_NoWinsor_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Outcome_Model_05_RF_NoWinsor_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Outcome_Model_05_RF_NoWinsor_v1_Predictions.csv"

# Candidate hyperparameter grids carried over from the locked core nonlinear stage
N_ESTIMATORS_GRID = [300, 500, 800]
MAX_DEPTH_GRID = [3, 5, 7, None]
MIN_SAMPLES_LEAF_GRID = [5, 20, 50]
MAX_FEATURES_GRID = ["sqrt", 0.33, 0.5]

# Fix randomness because random forests are stochastic
RF_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# Outcome-label safety checks
required_outcome_cols = ["outcome_mgmt_favorable", "outcome_dissident_favorable", "outcome_class"]
missing_outcome_cols = [c for c in required_outcome_cols if c not in df.columns]
if missing_outcome_cols:
    raise ValueError(f"Missing outcome safety columns: {missing_outcome_cols}")

if not (df["outcome_dissident_favorable"] == 1 - df["outcome_mgmt_favorable"]).all():
    raise ValueError("Outcome columns are not perfect complements.")



# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Final modeled predictor list used in estimation
# Note: this uses the operationalized categorical field ff17_for_model
# rather than the raw numeric ff17_code column retained for audit/whitelist checks.
all_predictors = continuous_features + binary_features + [categorical_feature]


# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }


def get_positive_class_probability(fitted_pipeline, X, positive_class=1):
    model = fitted_pipeline.named_steps["model"]
    class_list = list(model.classes_)
    if positive_class not in class_list:
        raise ValueError(
            f"Positive class {positive_class} not found in model.classes_: {class_list}"
        )
    positive_index = class_list.index(positive_class)
    return fitted_pipeline.predict_proba(X)[:, positive_index]

# =========================================================
# 6. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
):
    continuous_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],           # locked reference category
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=True,
        n_jobs=-1,
        random_state=RF_RANDOM_STATE,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf_model),
        ]
    )

# =========================================================
# 7. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 8. GRID SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []

total_combos = (
    len(N_ESTIMATORS_GRID)
    * len(MAX_DEPTH_GRID)
    * len(MIN_SAMPLES_LEAF_GRID)
    * len(MAX_FEATURES_GRID)
)
combo_counter = 0
search_start_time = time.time()

print("Starting N1 random forest tuning search...")
print(f"Total hyperparameter combinations: {total_combos}")
print(f"Total fold fits in tuning stage: {total_combos * len(cv_fold_defs)}")
print()

for n_estimators in N_ESTIMATORS_GRID:
    for max_depth in MAX_DEPTH_GRID:
        for min_samples_leaf in MIN_SAMPLES_LEAF_GRID:
            for max_features in MAX_FEATURES_GRID:
                combo_counter += 1
                combo_start_time = time.time()
                elapsed_minutes = (combo_start_time - search_start_time) / 60.0
                print(
                    f"[TUNING] Combo {combo_counter}/{total_combos} | "
                    f"elapsed={elapsed_minutes:.1f} min | "
                    f"n_estimators={n_estimators}, max_depth={max_depth}, "
                    f"min_samples_leaf={min_samples_leaf}, max_features={max_features}"
                )

                fold_rows = []

                for fold_def in cv_fold_defs:
                    fold_name = fold_def["fold"]
                    train_years = fold_def["train_years"]
                    val_years = fold_def["val_years"]

                    train_df = df[df["year"].isin(train_years)].copy()
                    val_df = df[df["year"].isin(val_years)].copy()

                    X_train = train_df[all_predictors].copy()
                    y_train = train_df[TARGET_COL].copy()

                    X_val = val_df[all_predictors].copy()
                    y_val = val_df[TARGET_COL].copy()

                    pipeline = build_pipeline(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        max_features=max_features,
                    )
                    pipeline.fit(X_train, y_train)
                    val_prob = pipeline.predict_proba(X_val)[:, 1]

                    fold_metrics = evaluate_predictions(y_val, val_prob)
                    fold_metrics.update(
                        {
                            "fold": fold_name,
                            "train_year_start": min(train_years),
                            "train_year_end": max(train_years),
                            "val_year_start": min(val_years),
                            "val_year_end": max(val_years),
                            "train_rows": len(train_df),
                            "train_positives": int(y_train.sum()),
                            "train_positive_rate": float(y_train.mean()),
                            "val_rows": len(val_df),
                            "val_positives": int(y_val.sum()),
                            "val_positive_rate": float(y_val.mean()),
                            "mean_predicted_probability": float(np.mean(val_prob)),
                        }
                    )
                    fold_rows.append(fold_metrics)

                fold_metrics_df = pd.DataFrame(fold_rows)

                combo_minutes = (time.time() - combo_start_time) / 60.0
                combo_mean_pr_auc = float(fold_metrics_df["pr_auc"].mean())
                combo_mean_roc_auc = float(fold_metrics_df["roc_auc"].mean())
                combo_mean_brier = float(fold_metrics_df["brier_score"].mean())

                tuning_rows.append(
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                        "max_features": max_features,
                        "cv_mean_pr_auc": combo_mean_pr_auc,
                        "cv_std_pr_auc": float(fold_metrics_df["pr_auc"].std(ddof=1)),
                        "cv_mean_roc_auc": combo_mean_roc_auc,
                        "cv_std_roc_auc": float(fold_metrics_df["roc_auc"].std(ddof=1)),
                        "cv_mean_brier_score": combo_mean_brier,
                        "cv_std_brier_score": float(fold_metrics_df["brier_score"].std(ddof=1)),
                    }
                )

                print(
                    f"[TUNING] Completed combo {combo_counter}/{total_combos} in {combo_minutes:.2f} min | "
                    f"cv_mean_pr_auc={combo_mean_pr_auc:.4f}, "
                    f"cv_mean_roc_auc={combo_mean_roc_auc:.4f}, "
                    f"cv_mean_brier={combo_mean_brier:.5f}"
                )
                print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler forest as a practical tie-break if metrics are essentially tied
#    (shallower depth, larger leaves, fewer trees)
depth_rank_map = {3: 0, 5: 1, 7: 2, None: 3}
max_features_rank_map = {"sqrt": 0, 0.33: 1, 0.5: 2}

tuning_summary_df["depth_rank"] = tuning_summary_df["max_depth"].map(depth_rank_map)
tuning_summary_df["max_features_rank"] = tuning_summary_df["max_features"].map(max_features_rank_map)

tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        "cv_mean_pr_auc",
        "cv_mean_roc_auc",
        "cv_mean_brier_score",
        "depth_rank",
        "min_samples_leaf",
        "n_estimators",
        "max_features_rank",
    ],
    ascending=[False, False, True, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, "n_estimators"])
selected_max_depth = tuning_summary_df.loc[0, "max_depth"]
if pd.isna(selected_max_depth):
    selected_max_depth = None
else:
    selected_max_depth = int(selected_max_depth)

selected_min_samples_leaf = int(tuning_summary_df.loc[0, "min_samples_leaf"])
selected_max_features = tuning_summary_df.loc[0, "max_features"]

# Restore numeric type for max_features when applicable
if isinstance(selected_max_features, str):
    if selected_max_features != "sqrt":
        selected_max_features = float(selected_max_features)

# =========================================================
# 9. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        min_samples_leaf=selected_min_samples_leaf,
        max_features=selected_max_features,
    )
    pipeline.fit(X_train, y_train)
    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_rf_nowinsor": val_prob,
                "selected_n_estimators": selected_n_estimators,
                "selected_max_depth": selected_max_depth,
                "selected_min_samples_leaf": selected_min_samples_leaf,
                "selected_max_features": selected_max_features,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 10. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()

X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    min_samples_leaf=selected_min_samples_leaf,
    max_features=selected_max_features,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = get_positive_class_probability(final_pipeline, X_test, positive_class=1)
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "deal_number": test_df["deal_number"].to_numpy(),
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "outcome_dissident_favorable": y_test.to_numpy(),
            "outcome_mgmt_favorable": test_df["outcome_mgmt_favorable"].to_numpy(),
            "outcome_class": test_df["outcome_class"].to_numpy(),
            "pred_outcome_rf_nowinsor": test_prob,
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 11. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].mean()),
            "roc_auc": float(selected_cv_df["roc_auc"].mean()),
            "brier_score": float(selected_cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(selected_cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(selected_cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

print("N1 random forest run complete.")
print(f"Saved tuning summary to: {TUNING_SUMMARY_CSV}")
print(f"Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print()
print("Selected hyperparameters:")
print(f"  n_estimators      = {selected_n_estimators}")
print(f"  max_depth         = {selected_max_depth}")
print(f"  min_samples_leaf  = {selected_min_samples_leaf}")
print(f"  max_features      = {selected_max_features}")

Starting N1 random forest tuning search...
Total hyperparameter combinations: 108
Total fold fits in tuning stage: 648

[TUNING] Combo 1/108 | elapsed=0.0 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=sqrt
[TUNING] Completed combo 1/108 in 0.04 min | cv_mean_pr_auc=0.2756, cv_mean_roc_auc=0.5682, cv_mean_brier=0.17517

[TUNING] Combo 2/108 | elapsed=0.0 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=0.33
[TUNING] Completed combo 2/108 in 0.04 min | cv_mean_pr_auc=0.2577, cv_mean_roc_auc=0.5461, cv_mean_brier=0.17366

[TUNING] Combo 3/108 | elapsed=0.1 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=0.5
[TUNING] Completed combo 3/108 in 0.04 min | cv_mean_pr_auc=0.2683, cv_mean_roc_auc=0.5570, cv_mean_brier=0.17260

[TUNING] Combo 4/108 | elapsed=0.1 min | n_estimators=300, max_depth=3, min_samples_leaf=20, max_features=sqrt
[TUNING] Completed combo 4/108 in 0.04 min | cv_mean_pr_auc=0.2760, cv_mean_roc_auc=0.5652, cv_

In [3]:
# Outcome Model 06 v1: Random forest, with winsorization
# Hyperparameters tuned by six-fold expanding-window CV through 2021
# Thesis-standard implementation under final outcome horse-race plan v1

import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Outcome_Master_Modeling_Panel_v1.csv"

TUNING_SUMMARY_CSV = BASE_DIR / "Outcome_Model_06_RF_Winsor_v1_Tuning_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Outcome_Model_06_RF_Winsor_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Outcome_Model_06_RF_Winsor_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Outcome_Model_06_RF_Winsor_v1_Predictions.csv"

# Candidate hyperparameter grids carried over from the locked core nonlinear stage
N_ESTIMATORS_GRID = [300, 500, 800]
MAX_DEPTH_GRID = [3, 5, 7, None]
MIN_SAMPLES_LEAF_GRID = [5, 20, 50]
MAX_FEATURES_GRID = ["sqrt", 0.33, 0.5]

# Fix randomness because random forests are stochastic
RF_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# Outcome-label safety checks
required_outcome_cols = ["outcome_mgmt_favorable", "outcome_dissident_favorable", "outcome_class"]
missing_outcome_cols = [c for c in required_outcome_cols if c not in df.columns]
if missing_outcome_cols:
    raise ValueError(f"Missing outcome safety columns: {missing_outcome_cols}")

if not (df["outcome_dissident_favorable"] == 1 - df["outcome_mgmt_favorable"]).all():
    raise ValueError("Outcome columns are not perfect complements.")



# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Final modeled predictor list used in estimation
# Note: this uses the operationalized categorical field ff17_for_model
# rather than the raw numeric ff17_code column retained for audit/whitelist checks.
all_predictors = continuous_features + binary_features + [categorical_feature]


# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }


def get_positive_class_probability(fitted_pipeline, X, positive_class=1):
    model = fitted_pipeline.named_steps["model"]
    class_list = list(model.classes_)
    if positive_class not in class_list:
        raise ValueError(
            f"Positive class {positive_class} not found in model.classes_: {class_list}"
        )
    positive_index = class_list.index(positive_class)
    return fitted_pipeline.predict_proba(X)[:, positive_index]

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
):
    continuous_transformer = Pipeline(
        steps=[
            ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],           # locked reference category
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=True,
        n_jobs=-1,
        random_state=RF_RANDOM_STATE,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf_model),
        ]
    )

# =========================================================
# 8. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 9. GRID SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []

total_combos = (
    len(N_ESTIMATORS_GRID)
    * len(MAX_DEPTH_GRID)
    * len(MIN_SAMPLES_LEAF_GRID)
    * len(MAX_FEATURES_GRID)
)
combo_counter = 0
search_start_time = time.time()

print("Starting N2 random forest tuning search...")
print(f"Total hyperparameter combinations: {total_combos}")
print(f"Total fold fits in tuning stage: {total_combos * len(cv_fold_defs)}")
print()

for n_estimators in N_ESTIMATORS_GRID:
    for max_depth in MAX_DEPTH_GRID:
        for min_samples_leaf in MIN_SAMPLES_LEAF_GRID:
            for max_features in MAX_FEATURES_GRID:
                combo_counter += 1
                combo_start_time = time.time()
                elapsed_minutes = (combo_start_time - search_start_time) / 60.0
                print(
                    f"[TUNING] Combo {combo_counter}/{total_combos} | "
                    f"elapsed={elapsed_minutes:.1f} min | "
                    f"n_estimators={n_estimators}, max_depth={max_depth}, "
                    f"min_samples_leaf={min_samples_leaf}, max_features={max_features}"
                )

                fold_rows = []

                for fold_def in cv_fold_defs:
                    fold_name = fold_def["fold"]
                    train_years = fold_def["train_years"]
                    val_years = fold_def["val_years"]

                    train_df = df[df["year"].isin(train_years)].copy()
                    val_df = df[df["year"].isin(val_years)].copy()

                    X_train = train_df[all_predictors].copy()
                    y_train = train_df[TARGET_COL].copy()

                    X_val = val_df[all_predictors].copy()
                    y_val = val_df[TARGET_COL].copy()

                    pipeline = build_pipeline(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        max_features=max_features,
                    )
                    pipeline.fit(X_train, y_train)
                    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

                    fold_metrics = evaluate_predictions(y_val, val_prob)
                    fold_metrics.update(
                        {
                            "fold": fold_name,
                            "train_year_start": min(train_years),
                            "train_year_end": max(train_years),
                            "val_year_start": min(val_years),
                            "val_year_end": max(val_years),
                            "train_rows": len(train_df),
                            "train_positives": int(y_train.sum()),
                            "train_positive_rate": float(y_train.mean()),
                            "val_rows": len(val_df),
                            "val_positives": int(y_val.sum()),
                            "val_positive_rate": float(y_val.mean()),
                            "mean_predicted_probability": float(np.mean(val_prob)),
                        }
                    )
                    fold_rows.append(fold_metrics)

                fold_metrics_df = pd.DataFrame(fold_rows)

                combo_minutes = (time.time() - combo_start_time) / 60.0
                combo_mean_pr_auc = float(fold_metrics_df["pr_auc"].mean())
                combo_mean_roc_auc = float(fold_metrics_df["roc_auc"].mean())
                combo_mean_brier = float(fold_metrics_df["brier_score"].mean())

                tuning_rows.append(
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                        "max_features": max_features,
                        "cv_mean_pr_auc": combo_mean_pr_auc,
                        "cv_std_pr_auc": float(fold_metrics_df["pr_auc"].std(ddof=1)),
                        "cv_mean_roc_auc": combo_mean_roc_auc,
                        "cv_std_roc_auc": float(fold_metrics_df["roc_auc"].std(ddof=1)),
                        "cv_mean_brier_score": combo_mean_brier,
                        "cv_std_brier_score": float(fold_metrics_df["brier_score"].std(ddof=1)),
                    }
                )

                print(
                    f"[TUNING] Completed combo {combo_counter}/{total_combos} in {combo_minutes:.2f} min | "
                    f"cv_mean_pr_auc={combo_mean_pr_auc:.4f}, "
                    f"cv_mean_roc_auc={combo_mean_roc_auc:.4f}, "
                    f"cv_mean_brier={combo_mean_brier:.5f}"
                )
                print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler forest as a practical tie-break if metrics are essentially tied
#    (shallower depth, larger leaves, fewer trees)
depth_rank_map = {3: 0, 5: 1, 7: 2, None: 3}
max_features_rank_map = {"sqrt": 0, 0.33: 1, 0.5: 2}

tuning_summary_df["depth_rank"] = tuning_summary_df["max_depth"].map(depth_rank_map)
tuning_summary_df["max_features_rank"] = tuning_summary_df["max_features"].map(max_features_rank_map)

tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        "cv_mean_pr_auc",
        "cv_mean_roc_auc",
        "cv_mean_brier_score",
        "depth_rank",
        "min_samples_leaf",
        "n_estimators",
        "max_features_rank",
    ],
    ascending=[False, False, True, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, "n_estimators"])
selected_max_depth = tuning_summary_df.loc[0, "max_depth"]
if pd.isna(selected_max_depth):
    selected_max_depth = None
else:
    selected_max_depth = int(selected_max_depth)

selected_min_samples_leaf = int(tuning_summary_df.loc[0, "min_samples_leaf"])
selected_max_features = tuning_summary_df.loc[0, "max_features"]

# Restore numeric type for max_features when applicable
if isinstance(selected_max_features, str):
    if selected_max_features != "sqrt":
        selected_max_features = float(selected_max_features)

# =========================================================
# 10. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        min_samples_leaf=selected_min_samples_leaf,
        max_features=selected_max_features,
    )
    pipeline.fit(X_train, y_train)
    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_rf_winsor": val_prob,
                "selected_n_estimators": selected_n_estimators,
                "selected_max_depth": selected_max_depth,
                "selected_min_samples_leaf": selected_min_samples_leaf,
                "selected_max_features": selected_max_features,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 11. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()

X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    min_samples_leaf=selected_min_samples_leaf,
    max_features=selected_max_features,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = get_positive_class_probability(final_pipeline, X_test, positive_class=1)
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "deal_number": test_df["deal_number"].to_numpy(),
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "outcome_dissident_favorable": y_test.to_numpy(),
            "outcome_mgmt_favorable": test_df["outcome_mgmt_favorable"].to_numpy(),
            "outcome_class": test_df["outcome_class"].to_numpy(),
            "pred_outcome_rf_winsor": test_prob,
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 12. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].mean()),
            "roc_auc": float(selected_cv_df["roc_auc"].mean()),
            "brier_score": float(selected_cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(selected_cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(selected_cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - search_start_time) / 60.0

print("N2 random forest run complete.")
print(f"Total runtime: {total_runtime_minutes:.2f} minutes")
print(f"Saved tuning summary to: {TUNING_SUMMARY_CSV}")
print(f"Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print()
print("Selected hyperparameters:")
print(f"  n_estimators      = {selected_n_estimators}")
print(f"  max_depth         = {selected_max_depth}")
print(f"  min_samples_leaf  = {selected_min_samples_leaf}")
print(f"  max_features      = {selected_max_features}")

Starting N2 random forest tuning search...
Total hyperparameter combinations: 108
Total fold fits in tuning stage: 648

[TUNING] Combo 1/108 | elapsed=0.0 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=sqrt
[TUNING] Completed combo 1/108 in 0.04 min | cv_mean_pr_auc=0.2786, cv_mean_roc_auc=0.5702, cv_mean_brier=0.17513

[TUNING] Combo 2/108 | elapsed=0.0 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=0.33
[TUNING] Completed combo 2/108 in 0.04 min | cv_mean_pr_auc=0.2590, cv_mean_roc_auc=0.5468, cv_mean_brier=0.17362

[TUNING] Combo 3/108 | elapsed=0.1 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=0.5
[TUNING] Completed combo 3/108 in 0.04 min | cv_mean_pr_auc=0.2687, cv_mean_roc_auc=0.5573, cv_mean_brier=0.17263

[TUNING] Combo 4/108 | elapsed=0.1 min | n_estimators=300, max_depth=3, min_samples_leaf=20, max_features=sqrt
[TUNING] Completed combo 4/108 in 0.04 min | cv_mean_pr_auc=0.2760, cv_mean_roc_auc=0.5652, cv_

In [2]:
# Outcome Model 07 v1: XGBoost, no winsorization
# Hyperparameters tuned by randomized search over an expanded discrete space
# under the final outcome horse-race plan v1 six-fold protocol.
# Thesis-standard implementation under the final outcome horse-race plan v1.
#
# IMPORTANT:
# - This script samples a fixed set of candidate configurations from the
#   expanded discrete search space using SEARCH_RANDOM_STATE.
# - Because N3 and N4 use the same parameter space, same candidate ordering,
#   and same SEARCH_RANDOM_STATE/N_RANDOM_CONFIGS, they evaluate the SAME
#   sampled hyperparameter configurations for fair comparison.

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path('.')
PANEL_PATH = BASE_DIR / 'Outcome_Master_Modeling_Panel_v1.csv'

SAMPLED_CONFIGS_CSV = BASE_DIR / 'Outcome_Model_07_XGB_NoWinsor_v1_Sampled_Configurations.csv'
TUNING_SUMMARY_CSV = BASE_DIR / 'Outcome_Model_07_XGB_NoWinsor_v1_Tuning_Summary.csv'
CV_FOLD_METRICS_CSV = BASE_DIR / 'Outcome_Model_07_XGB_NoWinsor_v1_SelectedHP_DevCV_Fold_Metrics.csv'
STAGE_METRICS_CSV = BASE_DIR / 'Outcome_Model_07_XGB_NoWinsor_v1_Stage_Metrics.csv'
PREDICTIONS_CSV = BASE_DIR / 'Outcome_Model_07_XGB_NoWinsor_v1_Predictions.csv'

# Expanded discrete search space (revised once after preliminary boundary-heavy runs)
N_ESTIMATORS_GRID = [200, 400, 800, 1200]
MAX_DEPTH_GRID = [2, 3, 4, 6]
LEARNING_RATE_GRID = [0.01, 0.03, 0.05, 0.10]
MIN_CHILD_WEIGHT_GRID = [1, 5, 10]
SUBSAMPLE_GRID = [0.6, 0.7, 0.85, 1.0]
COLSAMPLE_BYTREE_GRID = [0.5, 0.7, 0.85, 1.0]

# Randomized-search controls
N_RANDOM_CONFIGS = 300
SEARCH_RANDOM_STATE = 42

# Fix randomness because boosting is stochastic under subsampling
XGB_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# Outcome-label safety checks
required_outcome_cols = ["outcome_mgmt_favorable", "outcome_dissident_favorable", "outcome_class"]
missing_outcome_cols = [c for c in required_outcome_cols if c not in df.columns]
if missing_outcome_cols:
    raise ValueError(f"Missing outcome safety columns: {missing_outcome_cols}")

if not (df["outcome_dissident_favorable"] == 1 - df["outcome_mgmt_favorable"]).all():
    raise ValueError("Outcome columns are not perfect complements.")



# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Final modeled predictor list used in estimation
# Note: this uses the operationalized categorical field ff17_for_model
# rather than the raw numeric ff17_code column retained for audit/whitelist checks.
all_predictors = continuous_features + binary_features + [categorical_feature]


# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        'pr_auc': average_precision_score(y_true, y_prob),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'brier_score': brier_score_loss(y_true, y_prob),
    }


def get_positive_class_probability(fitted_pipeline, X, positive_class=1):
    model = fitted_pipeline.named_steps["model"]
    class_list = list(model.classes_)
    if positive_class not in class_list:
        raise ValueError(
            f"Positive class {positive_class} not found in model.classes_: {class_list}"
        )
    positive_index = class_list.index(positive_class)
    return fitted_pipeline.predict_proba(X)[:, positive_index]

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        lower_bounds = []
        upper_bounds = []
        for j in range(X.shape[1]):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))
        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth: int,
    learning_rate: float,
    min_child_weight: int,
    subsample: float,
    colsample_bytree: float,
):
    continuous_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unclassified')),
            ('onehot', OneHotEncoder(
                categories=[ff17_categories],
                drop=['Other'],
                handle_unknown='ignore',
                sparse_output=False,
            )),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', continuous_transformer, continuous_features),
            ('bin', binary_transformer, binary_features),
            ('ff17', categorical_transformer, [categorical_feature]),
        ],
        remainder='drop',
    )

    xgb_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', xgb_model),
        ]
    )

# =========================================================
# 8. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {'fold': 'fold_1', 'train_years': list(range(2010, 2016)), 'val_years': [2016]},
    {'fold': 'fold_2', 'train_years': list(range(2010, 2017)), 'val_years': [2017]},
    {'fold': 'fold_3', 'train_years': list(range(2010, 2018)), 'val_years': [2018]},
    {'fold': 'fold_4', 'train_years': list(range(2010, 2019)), 'val_years': [2019]},
    {'fold': 'fold_5', 'train_years': list(range(2010, 2020)), 'val_years': [2020]},
    {'fold': 'fold_6', 'train_years': list(range(2010, 2021)), 'val_years': [2021]},
]

# =========================================================
# 9. REPRODUCIBLE RANDOMIZED SEARCH OVER EXPANDED SPACE
# =========================================================
all_candidate_configs = [
    {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'min_child_weight': min_child_weight,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
    }
    for (n_estimators, max_depth, learning_rate, min_child_weight, subsample, colsample_bytree)
    in itertools.product(
        N_ESTIMATORS_GRID,
        MAX_DEPTH_GRID,
        LEARNING_RATE_GRID,
        MIN_CHILD_WEIGHT_GRID,
        SUBSAMPLE_GRID,
        COLSAMPLE_BYTREE_GRID,
    )
]

n_total_candidates = len(all_candidate_configs)
if N_RANDOM_CONFIGS > n_total_candidates:
    raise ValueError(
        f'N_RANDOM_CONFIGS={{N_RANDOM_CONFIGS}} exceeds total candidate count={{n_total_candidates}}.'
    )

rng = np.random.default_rng(SEARCH_RANDOM_STATE)
sampled_indices = rng.choice(n_total_candidates, size=N_RANDOM_CONFIGS, replace=False)
sampled_configs = [all_candidate_configs[int(i)] for i in sampled_indices]

sampled_configs_df = pd.DataFrame(sampled_configs)
sampled_configs_df.insert(0, 'sampled_config_rank', np.arange(1, len(sampled_configs_df) + 1))
sampled_configs_df.insert(1, 'original_candidate_index', sampled_indices.astype(int))
sampled_configs_df.to_csv(SAMPLED_CONFIGS_CSV, index=False)

# =========================================================
# 10. RANDOMIZED SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []
combo_counter = 0
search_start_time = time.time()

print(f"Starting N3 XGBoost randomized search...")
print(f'Total candidate combinations in expanded space: {n_total_candidates}')
print(f'Randomly sampled configurations to evaluate: {N_RANDOM_CONFIGS}')
print(f'Total fold fits in tuning stage: {N_RANDOM_CONFIGS * len(cv_fold_defs)}')
print(f'SEARCH_RANDOM_STATE: {SEARCH_RANDOM_STATE}')
print()

for cfg in sampled_configs:
    combo_counter += 1
    combo_start_time = time.time()
    elapsed_minutes = (combo_start_time - search_start_time) / 60.0

    n_estimators = int(cfg['n_estimators'])
    max_depth = int(cfg['max_depth'])
    learning_rate = float(cfg['learning_rate'])
    min_child_weight = int(cfg['min_child_weight'])
    subsample = float(cfg['subsample'])
    colsample_bytree = float(cfg['colsample_bytree'])

    print(
        f'[TUNING] Config {combo_counter}/{N_RANDOM_CONFIGS} | '
        f'elapsed={elapsed_minutes:.1f} min | '
        f'n_estimators={n_estimators}, max_depth={max_depth}, '
        f'learning_rate={learning_rate}, min_child_weight={min_child_weight}, '
        f'subsample={subsample}, colsample_bytree={colsample_bytree}'
    )

    fold_rows = []
    for fold_def in cv_fold_defs:
        fold_name = fold_def['fold']
        train_years = fold_def['train_years']
        val_years = fold_def['val_years']

        train_df = df[df['year'].isin(train_years)].copy()
        val_df = df[df['year'].isin(val_years)].copy()

        X_train = train_df[all_predictors].copy()
        y_train = train_df[TARGET_COL].copy()
        X_val = val_df[all_predictors].copy()
        y_val = val_df[TARGET_COL].copy()

        pipeline = build_pipeline(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            min_child_weight=min_child_weight,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
        )
        pipeline.fit(X_train, y_train)
        val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

        fold_metrics = evaluate_predictions(y_val, val_prob)
        fold_metrics.update(
            {
                'fold': fold_name,
                'train_year_start': min(train_years),
                'train_year_end': max(train_years),
                'val_year_start': min(val_years),
                'val_year_end': max(val_years),
                'train_rows': len(train_df),
                'train_positives': int(y_train.sum()),
                'train_positive_rate': float(y_train.mean()),
                'val_rows': len(val_df),
                'val_positives': int(y_val.sum()),
                'val_positive_rate': float(y_val.mean()),
                'mean_predicted_probability': float(np.mean(val_prob)),
            }
        )
        fold_rows.append(fold_metrics)

    fold_metrics_df = pd.DataFrame(fold_rows)
    combo_minutes = (time.time() - combo_start_time) / 60.0
    combo_mean_pr_auc = float(fold_metrics_df['pr_auc'].mean())
    combo_mean_roc_auc = float(fold_metrics_df['roc_auc'].mean())
    combo_mean_brier = float(fold_metrics_df['brier_score'].mean())

    tuning_rows.append(
        {
            'evaluated_config_rank': combo_counter,
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'learning_rate': learning_rate,
            'min_child_weight': min_child_weight,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'cv_mean_pr_auc': combo_mean_pr_auc,
            'cv_std_pr_auc': float(fold_metrics_df['pr_auc'].std(ddof=1)),
            'cv_mean_roc_auc': combo_mean_roc_auc,
            'cv_std_roc_auc': float(fold_metrics_df['roc_auc'].std(ddof=1)),
            'cv_mean_brier_score': combo_mean_brier,
            'cv_std_brier_score': float(fold_metrics_df['brier_score'].std(ddof=1)),
        }
    )

    print(
        f'[TUNING] Completed config {combo_counter}/{N_RANDOM_CONFIGS} in {combo_minutes:.2f} min | '
        f'cv_mean_pr_auc={combo_mean_pr_auc:.4f}, '
        f'cv_mean_roc_auc={combo_mean_roc_auc:.4f}, '
        f'cv_mean_brier={combo_mean_brier:.5f}'
    )
    print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler booster as a practical tie-break if metrics are essentially tied
#    (shallower depth, higher child weight, fewer trees, larger learning rate)
tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        'cv_mean_pr_auc',
        'cv_mean_roc_auc',
        'cv_mean_brier_score',
        'max_depth',
        'min_child_weight',
        'n_estimators',
        'learning_rate',
        'subsample',
        'colsample_bytree',
    ],
    ascending=[False, False, True, True, False, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, 'n_estimators'])
selected_max_depth = int(tuning_summary_df.loc[0, 'max_depth'])
selected_learning_rate = float(tuning_summary_df.loc[0, 'learning_rate'])
selected_min_child_weight = int(tuning_summary_df.loc[0, 'min_child_weight'])
selected_subsample = float(tuning_summary_df.loc[0, 'subsample'])
selected_colsample_bytree = float(tuning_summary_df.loc[0, 'colsample_bytree'])

# =========================================================
# 11. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def['fold']
    train_years = fold_def['train_years']
    val_years = fold_def['val_years']

    train_df = df[df['year'].isin(train_years)].copy()
    val_df = df[df['year'].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        learning_rate=selected_learning_rate,
        min_child_weight=selected_min_child_weight,
        subsample=selected_subsample,
        colsample_bytree=selected_colsample_bytree,
    )
    pipeline.fit(X_train, y_train)
    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'fold': fold_name,
            'train_year_start': min(train_years),
            'train_year_end': max(train_years),
            'val_year_start': min(val_years),
            'val_year_end': max(val_years),
            'train_rows': len(train_df),
            'train_positives': int(y_train.sum()),
            'train_positive_rate': float(y_train.mean()),
            'val_rows': len(val_df),
            'val_positives': int(y_val.sum()),
            'val_positive_rate': float(y_val.mean()),
            'mean_predicted_probability': float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_xgb_nowinsor": val_prob,
                "selected_n_estimators": selected_n_estimators,
                "selected_max_depth": selected_max_depth,
                "selected_learning_rate": selected_learning_rate,
                "selected_min_child_weight": selected_min_child_weight,
                "selected_subsample": selected_subsample,
                "selected_colsample_bytree": selected_colsample_bytree,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 12. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df['year'].between(2010, 2021)].copy()
test_df = df[df['year'].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    learning_rate=selected_learning_rate,
    min_child_weight=selected_min_child_weight,
    subsample=selected_subsample,
    colsample_bytree=selected_colsample_bytree,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = get_positive_class_probability(final_pipeline, X_test, positive_class=1)
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "deal_number": test_df["deal_number"].to_numpy(),
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "outcome_dissident_favorable": y_test.to_numpy(),
            "outcome_mgmt_favorable": test_df["outcome_mgmt_favorable"].to_numpy(),
            "outcome_class": test_df["outcome_class"].to_numpy(),
            "pred_outcome_xgb_nowinsor": test_prob,
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_learning_rate": selected_learning_rate,
            "selected_min_child_weight": selected_min_child_weight,
            "selected_subsample": selected_subsample,
            "selected_colsample_bytree": selected_colsample_bytree,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 13. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            'stage': 'development_cv_mean',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].mean()),
            'roc_auc': float(selected_cv_df['roc_auc'].mean()),
            'brier_score': float(selected_cv_df['brier_score'].mean()),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'development_cv_std',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].std(ddof=1)),
            'roc_auc': float(selected_cv_df['roc_auc'].std(ddof=1)),
            'brier_score': float(selected_cv_df['brier_score'].std(ddof=1)),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'final_test',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(test_metrics['pr_auc']),
            'roc_auc': float(test_metrics['roc_auc']),
            'brier_score': float(test_metrics['brier_score']),
            'rows': int(len(test_df)),
            'positives': int(y_test.sum()),
            'positive_rate': float(y_test.mean()),
            'mean_predicted_probability': float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - search_start_time) / 60.0

print(f"N3 XGBoost randomized-search run complete.")
print(f'Total runtime: {total_runtime_minutes:.2f} minutes')
print(f'Saved sampled configs to: {SAMPLED_CONFIGS_CSV}')
print(f'Saved tuning summary to: {TUNING_SUMMARY_CSV}')
print(f'Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}')
print(f'Saved stage metrics to: {STAGE_METRICS_CSV}')
print(f'Saved predictions to: {PREDICTIONS_CSV}')
print()
print('Selected hyperparameters:')
print(f'  n_estimators      = {selected_n_estimators}')
print(f'  max_depth         = {selected_max_depth}')
print(f'  learning_rate     = {selected_learning_rate}')
print(f'  min_child_weight  = {selected_min_child_weight}')
print(f'  subsample         = {selected_subsample}')
print(f'  colsample_bytree  = {selected_colsample_bytree}')

Starting N3 XGBoost randomized search...
Total candidate combinations in expanded space: 3072
Randomly sampled configurations to evaluate: 300
Total fold fits in tuning stage: 1800
SEARCH_RANDOM_STATE: 42

[TUNING] Config 1/300 | elapsed=0.0 min | n_estimators=400, max_depth=6, learning_rate=0.03, min_child_weight=10, subsample=0.7, colsample_bytree=1.0
[TUNING] Completed config 1/300 in 0.14 min | cv_mean_pr_auc=0.2732, cv_mean_roc_auc=0.5153, cv_mean_brier=0.17888

[TUNING] Config 2/300 | elapsed=0.1 min | n_estimators=800, max_depth=6, learning_rate=0.05, min_child_weight=10, subsample=0.85, colsample_bytree=1.0
[TUNING] Completed config 2/300 in 0.15 min | cv_mean_pr_auc=0.2708, cv_mean_roc_auc=0.4991, cv_mean_brier=0.19030

[TUNING] Config 3/300 | elapsed=0.3 min | n_estimators=200, max_depth=3, learning_rate=0.1, min_child_weight=10, subsample=1.0, colsample_bytree=0.7
[TUNING] Completed config 3/300 in 0.02 min | cv_mean_pr_auc=0.2512, cv_mean_roc_auc=0.5135, cv_mean_brier=0.183

In [1]:
# Outcome Model 08 v1: XGBoost, with winsorization
# Hyperparameters tuned by randomized search over an expanded discrete space
# under the final outcome horse-race plan v1 six-fold protocol.
# Thesis-standard implementation under the final outcome horse-race plan v1.
#
# IMPORTANT:
# - This script samples a fixed set of candidate configurations from the
#   expanded discrete search space using SEARCH_RANDOM_STATE.
# - Because N3 and N4 use the same parameter space, same candidate ordering,
#   and same SEARCH_RANDOM_STATE/N_RANDOM_CONFIGS, they evaluate the SAME
#   sampled hyperparameter configurations for fair comparison.

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path('.')
PANEL_PATH = BASE_DIR / 'Outcome_Master_Modeling_Panel_v1.csv'

SAMPLED_CONFIGS_CSV = BASE_DIR / 'Outcome_Model_08_XGB_Winsor_v1_Sampled_Configurations.csv'
TUNING_SUMMARY_CSV = BASE_DIR / 'Outcome_Model_08_XGB_Winsor_v1_Tuning_Summary.csv'
CV_FOLD_METRICS_CSV = BASE_DIR / 'Outcome_Model_08_XGB_Winsor_v1_SelectedHP_DevCV_Fold_Metrics.csv'
STAGE_METRICS_CSV = BASE_DIR / 'Outcome_Model_08_XGB_Winsor_v1_Stage_Metrics.csv'
PREDICTIONS_CSV = BASE_DIR / 'Outcome_Model_08_XGB_Winsor_v1_Predictions.csv'

# Expanded discrete search space (revised once after preliminary boundary-heavy runs)
N_ESTIMATORS_GRID = [200, 400, 800, 1200]
MAX_DEPTH_GRID = [2, 3, 4, 6]
LEARNING_RATE_GRID = [0.01, 0.03, 0.05, 0.10]
MIN_CHILD_WEIGHT_GRID = [1, 5, 10]
SUBSAMPLE_GRID = [0.6, 0.7, 0.85, 1.0]
COLSAMPLE_BYTREE_GRID = [0.5, 0.7, 0.85, 1.0]

# Randomized-search controls
N_RANDOM_CONFIGS = 300
SEARCH_RANDOM_STATE = 42

# Fix randomness because boosting is stochastic under subsampling
XGB_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# Outcome-label safety checks
required_outcome_cols = ["outcome_mgmt_favorable", "outcome_dissident_favorable", "outcome_class"]
missing_outcome_cols = [c for c in required_outcome_cols if c not in df.columns]
if missing_outcome_cols:
    raise ValueError(f"Missing outcome safety columns: {missing_outcome_cols}")

if not (df["outcome_dissident_favorable"] == 1 - df["outcome_mgmt_favorable"]).all():
    raise ValueError("Outcome columns are not perfect complements.")



# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Final modeled predictor list used in estimation
# Note: this uses the operationalized categorical field ff17_for_model
# rather than the raw numeric ff17_code column retained for audit/whitelist checks.
all_predictors = continuous_features + binary_features + [categorical_feature]


# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        'pr_auc': average_precision_score(y_true, y_prob),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'brier_score': brier_score_loss(y_true, y_prob),
    }


def get_positive_class_probability(fitted_pipeline, X, positive_class=1):
    model = fitted_pipeline.named_steps["model"]
    class_list = list(model.classes_)
    if positive_class not in class_list:
        raise ValueError(
            f"Positive class {positive_class} not found in model.classes_: {class_list}"
        )
    positive_index = class_list.index(positive_class)
    return fitted_pipeline.predict_proba(X)[:, positive_index]

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        lower_bounds = []
        upper_bounds = []
        for j in range(X.shape[1]):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))
        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth: int,
    learning_rate: float,
    min_child_weight: int,
    subsample: float,
    colsample_bytree: float,
):
    continuous_transformer = Pipeline(
        steps=[
            ('winsorizer', PercentileWinsorizer(lower=1.0, upper=99.0)),
            ('imputer', SimpleImputer(strategy='median')),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unclassified')),
            ('onehot', OneHotEncoder(
                categories=[ff17_categories],
                drop=['Other'],
                handle_unknown='ignore',
                sparse_output=False,
            )),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', continuous_transformer, continuous_features),
            ('bin', binary_transformer, binary_features),
            ('ff17', categorical_transformer, [categorical_feature]),
        ],
        remainder='drop',
    )

    xgb_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', xgb_model),
        ]
    )

# =========================================================
# 8. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {'fold': 'fold_1', 'train_years': list(range(2010, 2016)), 'val_years': [2016]},
    {'fold': 'fold_2', 'train_years': list(range(2010, 2017)), 'val_years': [2017]},
    {'fold': 'fold_3', 'train_years': list(range(2010, 2018)), 'val_years': [2018]},
    {'fold': 'fold_4', 'train_years': list(range(2010, 2019)), 'val_years': [2019]},
    {'fold': 'fold_5', 'train_years': list(range(2010, 2020)), 'val_years': [2020]},
    {'fold': 'fold_6', 'train_years': list(range(2010, 2021)), 'val_years': [2021]},
]

# =========================================================
# 9. REPRODUCIBLE RANDOMIZED SEARCH OVER EXPANDED SPACE
# =========================================================
all_candidate_configs = [
    {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'min_child_weight': min_child_weight,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
    }
    for (n_estimators, max_depth, learning_rate, min_child_weight, subsample, colsample_bytree)
    in itertools.product(
        N_ESTIMATORS_GRID,
        MAX_DEPTH_GRID,
        LEARNING_RATE_GRID,
        MIN_CHILD_WEIGHT_GRID,
        SUBSAMPLE_GRID,
        COLSAMPLE_BYTREE_GRID,
    )
]

n_total_candidates = len(all_candidate_configs)
if N_RANDOM_CONFIGS > n_total_candidates:
    raise ValueError(
        f'N_RANDOM_CONFIGS={{N_RANDOM_CONFIGS}} exceeds total candidate count={{n_total_candidates}}.'
    )

rng = np.random.default_rng(SEARCH_RANDOM_STATE)
sampled_indices = rng.choice(n_total_candidates, size=N_RANDOM_CONFIGS, replace=False)
sampled_configs = [all_candidate_configs[int(i)] for i in sampled_indices]

sampled_configs_df = pd.DataFrame(sampled_configs)
sampled_configs_df.insert(0, 'sampled_config_rank', np.arange(1, len(sampled_configs_df) + 1))
sampled_configs_df.insert(1, 'original_candidate_index', sampled_indices.astype(int))
sampled_configs_df.to_csv(SAMPLED_CONFIGS_CSV, index=False)

# =========================================================
# 10. RANDOMIZED SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []
combo_counter = 0
search_start_time = time.time()

print(f"Starting N4 XGBoost randomized search...")
print(f'Total candidate combinations in expanded space: {n_total_candidates}')
print(f'Randomly sampled configurations to evaluate: {N_RANDOM_CONFIGS}')
print(f'Total fold fits in tuning stage: {N_RANDOM_CONFIGS * len(cv_fold_defs)}')
print(f'SEARCH_RANDOM_STATE: {SEARCH_RANDOM_STATE}')
print()

for cfg in sampled_configs:
    combo_counter += 1
    combo_start_time = time.time()
    elapsed_minutes = (combo_start_time - search_start_time) / 60.0

    n_estimators = int(cfg['n_estimators'])
    max_depth = int(cfg['max_depth'])
    learning_rate = float(cfg['learning_rate'])
    min_child_weight = int(cfg['min_child_weight'])
    subsample = float(cfg['subsample'])
    colsample_bytree = float(cfg['colsample_bytree'])

    print(
        f'[TUNING] Config {combo_counter}/{N_RANDOM_CONFIGS} | '
        f'elapsed={elapsed_minutes:.1f} min | '
        f'n_estimators={n_estimators}, max_depth={max_depth}, '
        f'learning_rate={learning_rate}, min_child_weight={min_child_weight}, '
        f'subsample={subsample}, colsample_bytree={colsample_bytree}'
    )

    fold_rows = []
    for fold_def in cv_fold_defs:
        fold_name = fold_def['fold']
        train_years = fold_def['train_years']
        val_years = fold_def['val_years']

        train_df = df[df['year'].isin(train_years)].copy()
        val_df = df[df['year'].isin(val_years)].copy()

        X_train = train_df[all_predictors].copy()
        y_train = train_df[TARGET_COL].copy()
        X_val = val_df[all_predictors].copy()
        y_val = val_df[TARGET_COL].copy()

        pipeline = build_pipeline(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            min_child_weight=min_child_weight,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
        )
        pipeline.fit(X_train, y_train)
        val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

        fold_metrics = evaluate_predictions(y_val, val_prob)
        fold_metrics.update(
            {
                'fold': fold_name,
                'train_year_start': min(train_years),
                'train_year_end': max(train_years),
                'val_year_start': min(val_years),
                'val_year_end': max(val_years),
                'train_rows': len(train_df),
                'train_positives': int(y_train.sum()),
                'train_positive_rate': float(y_train.mean()),
                'val_rows': len(val_df),
                'val_positives': int(y_val.sum()),
                'val_positive_rate': float(y_val.mean()),
                'mean_predicted_probability': float(np.mean(val_prob)),
            }
        )
        fold_rows.append(fold_metrics)

    fold_metrics_df = pd.DataFrame(fold_rows)
    combo_minutes = (time.time() - combo_start_time) / 60.0
    combo_mean_pr_auc = float(fold_metrics_df['pr_auc'].mean())
    combo_mean_roc_auc = float(fold_metrics_df['roc_auc'].mean())
    combo_mean_brier = float(fold_metrics_df['brier_score'].mean())

    tuning_rows.append(
        {
            'evaluated_config_rank': combo_counter,
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'learning_rate': learning_rate,
            'min_child_weight': min_child_weight,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'cv_mean_pr_auc': combo_mean_pr_auc,
            'cv_std_pr_auc': float(fold_metrics_df['pr_auc'].std(ddof=1)),
            'cv_mean_roc_auc': combo_mean_roc_auc,
            'cv_std_roc_auc': float(fold_metrics_df['roc_auc'].std(ddof=1)),
            'cv_mean_brier_score': combo_mean_brier,
            'cv_std_brier_score': float(fold_metrics_df['brier_score'].std(ddof=1)),
        }
    )

    print(
        f'[TUNING] Completed config {combo_counter}/{N_RANDOM_CONFIGS} in {combo_minutes:.2f} min | '
        f'cv_mean_pr_auc={combo_mean_pr_auc:.4f}, '
        f'cv_mean_roc_auc={combo_mean_roc_auc:.4f}, '
        f'cv_mean_brier={combo_mean_brier:.5f}'
    )
    print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler booster as a practical tie-break if metrics are essentially tied
#    (shallower depth, higher child weight, fewer trees, larger learning rate)
tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        'cv_mean_pr_auc',
        'cv_mean_roc_auc',
        'cv_mean_brier_score',
        'max_depth',
        'min_child_weight',
        'n_estimators',
        'learning_rate',
        'subsample',
        'colsample_bytree',
    ],
    ascending=[False, False, True, True, False, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, 'n_estimators'])
selected_max_depth = int(tuning_summary_df.loc[0, 'max_depth'])
selected_learning_rate = float(tuning_summary_df.loc[0, 'learning_rate'])
selected_min_child_weight = int(tuning_summary_df.loc[0, 'min_child_weight'])
selected_subsample = float(tuning_summary_df.loc[0, 'subsample'])
selected_colsample_bytree = float(tuning_summary_df.loc[0, 'colsample_bytree'])

# =========================================================
# 11. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def['fold']
    train_years = fold_def['train_years']
    val_years = fold_def['val_years']

    train_df = df[df['year'].isin(train_years)].copy()
    val_df = df[df['year'].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        learning_rate=selected_learning_rate,
        min_child_weight=selected_min_child_weight,
        subsample=selected_subsample,
        colsample_bytree=selected_colsample_bytree,
    )
    pipeline.fit(X_train, y_train)
    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'fold': fold_name,
            'train_year_start': min(train_years),
            'train_year_end': max(train_years),
            'val_year_start': min(val_years),
            'val_year_end': max(val_years),
            'train_rows': len(train_df),
            'train_positives': int(y_train.sum()),
            'train_positive_rate': float(y_train.mean()),
            'val_rows': len(val_df),
            'val_positives': int(y_val.sum()),
            'val_positive_rate': float(y_val.mean()),
            'mean_predicted_probability': float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_xgb_winsor": val_prob,
                "selected_n_estimators": selected_n_estimators,
                "selected_max_depth": selected_max_depth,
                "selected_learning_rate": selected_learning_rate,
                "selected_min_child_weight": selected_min_child_weight,
                "selected_subsample": selected_subsample,
                "selected_colsample_bytree": selected_colsample_bytree,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 12. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df['year'].between(2010, 2021)].copy()
test_df = df[df['year'].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    learning_rate=selected_learning_rate,
    min_child_weight=selected_min_child_weight,
    subsample=selected_subsample,
    colsample_bytree=selected_colsample_bytree,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = get_positive_class_probability(final_pipeline, X_test, positive_class=1)
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "deal_number": test_df["deal_number"].to_numpy(),
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "outcome_dissident_favorable": y_test.to_numpy(),
            "outcome_mgmt_favorable": test_df["outcome_mgmt_favorable"].to_numpy(),
            "outcome_class": test_df["outcome_class"].to_numpy(),
            "pred_outcome_xgb_winsor": test_prob,
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_learning_rate": selected_learning_rate,
            "selected_min_child_weight": selected_min_child_weight,
            "selected_subsample": selected_subsample,
            "selected_colsample_bytree": selected_colsample_bytree,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 13. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            'stage': 'development_cv_mean',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].mean()),
            'roc_auc': float(selected_cv_df['roc_auc'].mean()),
            'brier_score': float(selected_cv_df['brier_score'].mean()),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'development_cv_std',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].std(ddof=1)),
            'roc_auc': float(selected_cv_df['roc_auc'].std(ddof=1)),
            'brier_score': float(selected_cv_df['brier_score'].std(ddof=1)),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'final_test',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(test_metrics['pr_auc']),
            'roc_auc': float(test_metrics['roc_auc']),
            'brier_score': float(test_metrics['brier_score']),
            'rows': int(len(test_df)),
            'positives': int(y_test.sum()),
            'positive_rate': float(y_test.mean()),
            'mean_predicted_probability': float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - search_start_time) / 60.0

print(f"N4 XGBoost randomized-search run complete.")
print(f'Total runtime: {total_runtime_minutes:.2f} minutes')
print(f'Saved sampled configs to: {SAMPLED_CONFIGS_CSV}')
print(f'Saved tuning summary to: {TUNING_SUMMARY_CSV}')
print(f'Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}')
print(f'Saved stage metrics to: {STAGE_METRICS_CSV}')
print(f'Saved predictions to: {PREDICTIONS_CSV}')
print()
print('Selected hyperparameters:')
print(f'  n_estimators      = {selected_n_estimators}')
print(f'  max_depth         = {selected_max_depth}')
print(f'  learning_rate     = {selected_learning_rate}')
print(f'  min_child_weight  = {selected_min_child_weight}')
print(f'  subsample         = {selected_subsample}')
print(f'  colsample_bytree  = {selected_colsample_bytree}')

Starting N4 XGBoost randomized search...
Total candidate combinations in expanded space: 3072
Randomly sampled configurations to evaluate: 300
Total fold fits in tuning stage: 1800
SEARCH_RANDOM_STATE: 42

[TUNING] Config 1/300 | elapsed=0.0 min | n_estimators=400, max_depth=6, learning_rate=0.03, min_child_weight=10, subsample=0.7, colsample_bytree=1.0
[TUNING] Completed config 1/300 in 0.06 min | cv_mean_pr_auc=0.2721, cv_mean_roc_auc=0.5056, cv_mean_brier=0.17972

[TUNING] Config 2/300 | elapsed=0.1 min | n_estimators=800, max_depth=6, learning_rate=0.05, min_child_weight=10, subsample=0.85, colsample_bytree=1.0
[TUNING] Completed config 2/300 in 0.10 min | cv_mean_pr_auc=0.2606, cv_mean_roc_auc=0.4916, cv_mean_brier=0.19092

[TUNING] Config 3/300 | elapsed=0.2 min | n_estimators=200, max_depth=3, learning_rate=0.1, min_child_weight=10, subsample=1.0, colsample_bytree=0.7
[TUNING] Completed config 3/300 in 0.01 min | cv_mean_pr_auc=0.2505, cv_mean_roc_auc=0.5072, cv_mean_brier=0.183

In [3]:
# ============================================================
# Mature-sample sensitivity for outcome extension
# N3 only: XGBoost No Winsor, fixed hyperparameters
# Sample: 2010-2021
# CV: 2010-2014->2015, 2010-2015->2016, 2010-2016->2017, 2010-2017->2018
# Final holdout: 2019-2021
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# -----------------------------
# Safety checks
# -----------------------------
assert TARGET_COL == "outcome_dissident_favorable", f"Unexpected TARGET_COL: {TARGET_COL}"
assert "year" in df.columns
assert "deal_number" in df.columns
assert TARGET_COL in df.columns
assert all(col in df.columns for col in all_predictors), "Some predictors are missing from df."

print("Running mature-sample sensitivity on N3 only.")
print("TARGET_COL:", TARGET_COL)
print("Total rows in full outcome panel:", len(df))

# -----------------------------
# Restrict to mature sample
# -----------------------------
mature_df = df[df["year"].between(2010, 2021)].copy()

print("\nRows in mature sample (2010-2021):", len(mature_df))
print("Years in mature sample:", sorted(mature_df["year"].unique()))

# Year-by-year summary for documentation
year_summary = (
    mature_df.groupby("year")[TARGET_COL]
    .agg(
        n_campaigns="size",
        dissident_favorable_count="sum",
        dissident_favorable_rate="mean"
    )
    .reset_index()
)
print("\nYear summary:")
print(year_summary)

# -----------------------------
# Sensitivity split design
# -----------------------------
fold_definitions = [
    {"fold": "fold_1", "train_years": list(range(2010, 2015)), "val_years": [2015]},
    {"fold": "fold_2", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_3", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_4", "train_years": list(range(2010, 2018)), "val_years": [2018]},
]

final_train_years = list(range(2010, 2019))
final_holdout_years = [2019, 2020, 2021]

split_summary_rows = []
for spec in fold_definitions:
    train_df_tmp = mature_df[mature_df["year"].isin(spec["train_years"])].copy()
    val_df_tmp = mature_df[mature_df["year"].isin(spec["val_years"])].copy()

    split_summary_rows.append({
        "split_name": spec["fold"],
        "train_years": f"{min(spec['train_years'])}-{max(spec['train_years'])}",
        "validation_years": ",".join(map(str, spec["val_years"])),
        "n_train": len(train_df_tmp),
        "train_positive_rate": train_df_tmp[TARGET_COL].mean(),
        "n_validation": len(val_df_tmp),
        "validation_positive_rate": val_df_tmp[TARGET_COL].mean(),
    })

# Final holdout summary row
final_train_df_tmp = mature_df[mature_df["year"].isin(final_train_years)].copy()
final_holdout_df_tmp = mature_df[mature_df["year"].isin(final_holdout_years)].copy()
split_summary_rows.append({
    "split_name": "final_holdout",
    "train_years": f"{min(final_train_years)}-{max(final_train_years)}",
    "validation_years": ",".join(map(str, final_holdout_years)),
    "n_train": len(final_train_df_tmp),
    "train_positive_rate": final_train_df_tmp[TARGET_COL].mean(),
    "n_validation": len(final_holdout_df_tmp),
    "validation_positive_rate": final_holdout_df_tmp[TARGET_COL].mean(),
})

split_summary_df = pd.DataFrame(split_summary_rows)
print("\nSplit summary:")
print(split_summary_df)

# -----------------------------
# Fixed N3 hyperparameters
# -----------------------------
xgb_params_n3 = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "subsample": 1.0,
    "colsample_bytree": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

# -----------------------------
# Helper functions
# -----------------------------
def build_n3_pipeline():
    """
    Build a fresh pipeline for each fold/final fit.
    Matches the N3 no-winsor preprocessing philosophy.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("continuous", SimpleImputer(strategy="median"), continuous_features),
            ("binary", SimpleImputer(strategy="constant", fill_value=0), binary_features),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                [categorical_feature],
            ),
        ],
        remainder="drop",
    )

    model = XGBClassifier(**xgb_params_n3)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    return pipeline

def get_positive_class_probability(fitted_pipeline, X, positive_class=1):
    model = fitted_pipeline.named_steps["model"]
    class_list = list(model.classes_)
    if positive_class not in class_list:
        raise ValueError(f"Positive class {positive_class} not found in model.classes_: {class_list}")
    positive_index = class_list.index(positive_class)
    return fitted_pipeline.predict_proba(X)[:, positive_index]

def safe_roc_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def compute_metric_row(model_id, evaluation_stage, fold_name, y_true, y_prob, n_obs):
    return {
        "model_id": model_id,
        "evaluation_stage": evaluation_stage,
        "fold": fold_name,
        "n_obs": int(n_obs),
        "positive_rate": float(np.mean(y_true)),
        "mean_predicted_probability": float(np.mean(y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "roc_auc": float(safe_roc_auc(y_true, y_prob)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
    }

# -----------------------------
# Run fold-by-fold CV
# -----------------------------
MODEL_ID = "N3_MatureSample_2010_2021_Holdout2019_2021"
PRED_COL = "pred_outcome_xgb_nowinsor_mature"

fold_metric_rows = []
prediction_rows = []

print("\nStarting mature-sample sensitivity CV for N3...")
print(f"Fixed hyperparameters: {xgb_params_n3}")

for spec in fold_definitions:
    fold_name = spec["fold"]
    train_df = mature_df[mature_df["year"].isin(spec["train_years"])].copy()
    val_df = mature_df[mature_df["year"].isin(spec["val_years"])].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_n3_pipeline()
    pipeline.fit(X_train, y_train)

    model = pipeline.named_steps["model"]
    print(f"\n[{fold_name}] model.classes_ = {model.classes_}")
    assert list(model.classes_) == [0, 1], f"Unexpected class ordering: {model.classes_}"

    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

    mean_prob_y1 = float(np.mean(val_prob[y_val.to_numpy() == 1])) if (y_val == 1).any() else np.nan
    mean_prob_y0 = float(np.mean(val_prob[y_val.to_numpy() == 0])) if (y_val == 0).any() else np.nan

    print(f"[{fold_name}] train years: {min(spec['train_years'])}-{max(spec['train_years'])}")
    print(f"[{fold_name}] validation years: {spec['val_years']}")
    print(f"[{fold_name}] n_train = {len(train_df)}, train positive rate = {y_train.mean():.4f}")
    print(f"[{fold_name}] n_val = {len(val_df)}, val positive rate = {y_val.mean():.4f}")
    print(f"[{fold_name}] mean prob | y=1 = {mean_prob_y1:.6f}")
    print(f"[{fold_name}] mean prob | y=0 = {mean_prob_y0:.6f}")

    fold_metric_rows.append(
        compute_metric_row(
            model_id=MODEL_ID,
            evaluation_stage="dev_cv",
            fold_name=fold_name,
            y_true=y_val,
            y_prob=val_prob,
            n_obs=len(val_df),
        )
    )

    pred_df = val_df[[
        "deal_number", "year", "outcome_mgmt_favorable",
        "outcome_dissident_favorable", "outcome_class"
    ]].copy()
    pred_df["model_id"] = MODEL_ID
    pred_df["evaluation_stage"] = "dev_cv"
    pred_df["fold"] = fold_name
    pred_df[PRED_COL] = val_prob
    prediction_rows.append(pred_df)

fold_metrics_df = pd.DataFrame(fold_metric_rows)
print("\nFold metrics:")
print(fold_metrics_df)

# -----------------------------
# Final mature holdout
# -----------------------------
print("\nRefitting N3 on 2010-2018 and evaluating on mature holdout 2019-2021...")

dev_df = mature_df[mature_df["year"].isin(final_train_years)].copy()
holdout_df = mature_df[mature_df["year"].isin(final_holdout_years)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()

X_holdout = holdout_df[all_predictors].copy()
y_holdout = holdout_df[TARGET_COL].copy()

final_pipeline = build_n3_pipeline()
final_pipeline.fit(X_dev, y_dev)

final_model = final_pipeline.named_steps["model"]
print("[final_holdout] model.classes_ =", final_model.classes_)
assert list(final_model.classes_) == [0, 1], f"Unexpected class ordering: {final_model.classes_}"

holdout_prob = get_positive_class_probability(final_pipeline, X_holdout, positive_class=1)

holdout_mean_prob_y1 = float(np.mean(holdout_prob[y_holdout.to_numpy() == 1])) if (y_holdout == 1).any() else np.nan
holdout_mean_prob_y0 = float(np.mean(holdout_prob[y_holdout.to_numpy() == 0])) if (y_holdout == 0).any() else np.nan

print(f"[final_holdout] n_train = {len(dev_df)}, train positive rate = {y_dev.mean():.4f}")
print(f"[final_holdout] n_holdout = {len(holdout_df)}, holdout positive rate = {y_holdout.mean():.4f}")
print(f"[final_holdout] mean prob | y=1 = {holdout_mean_prob_y1:.6f}")
print(f"[final_holdout] mean prob | y=0 = {holdout_mean_prob_y0:.6f}")

final_stage_row = compute_metric_row(
    model_id=MODEL_ID,
    evaluation_stage="final_holdout",
    fold_name="2019_2021_holdout",
    y_true=y_holdout,
    y_prob=holdout_prob,
    n_obs=len(holdout_df),
)

# Prediction rows for holdout
holdout_pred_df = holdout_df[[
    "deal_number", "year", "outcome_mgmt_favorable",
    "outcome_dissident_favorable", "outcome_class"
]].copy()
holdout_pred_df["model_id"] = MODEL_ID
holdout_pred_df["evaluation_stage"] = "final_holdout"
holdout_pred_df["fold"] = "2019_2021_holdout"
holdout_pred_df[PRED_COL] = holdout_prob
prediction_rows.append(holdout_pred_df)

predictions_df = pd.concat(prediction_rows, axis=0, ignore_index=True)

# -----------------------------
# Stage summary
# -----------------------------
dev_cv_summary = {
    "model_id": MODEL_ID,
    "evaluation_stage": "dev_cv_mean",
    "fold": "all_folds",
    "n_obs": int(fold_metrics_df["n_obs"].sum()),
    "positive_rate": float(np.average(
        fold_metrics_df["positive_rate"],
        weights=fold_metrics_df["n_obs"]
    )),
    "mean_predicted_probability": float(np.average(
        fold_metrics_df["mean_predicted_probability"],
        weights=fold_metrics_df["n_obs"]
    )),
    "pr_auc": float(fold_metrics_df["pr_auc"].mean()),
    "roc_auc": float(fold_metrics_df["roc_auc"].mean()),
    "brier_score": float(fold_metrics_df["brier_score"].mean()),
}

stage_metrics_df = pd.DataFrame([dev_cv_summary, final_stage_row])

print("\nStage metrics:")
print(stage_metrics_df)

# -----------------------------
# Save outputs
# -----------------------------
output_prefix = "Outcome_Model_07_XGB_NoWinsor_v1_MatureSample_2010_2021_Holdout2019_2021"

fold_metrics_path = Path(f"{output_prefix}_Fold_Metrics.csv")
stage_metrics_path = Path(f"{output_prefix}_Stage_Metrics.csv")
predictions_path = Path(f"{output_prefix}_Predictions.csv")
year_summary_path = Path(f"{output_prefix}_Year_Summary.csv")
split_summary_path = Path(f"{output_prefix}_Split_Summary.csv")

fold_metrics_df.to_csv(fold_metrics_path, index=False)
stage_metrics_df.to_csv(stage_metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
year_summary.to_csv(year_summary_path, index=False)
split_summary_df.to_csv(split_summary_path, index=False)

print("\nSaved files:")
print(" -", fold_metrics_path)
print(" -", stage_metrics_path)
print(" -", predictions_path)
print(" -", year_summary_path)
print(" -", split_summary_path)

print("\nMature-sample sensitivity run complete.")

Running mature-sample sensitivity on N3 only.
TARGET_COL: outcome_dissident_favorable
Total rows in full outcome panel: 1401

Rows in mature sample (2010-2021): 1168
Years in mature sample: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

Year summary:
    year  n_campaigns  dissident_favorable_count  dissident_favorable_rate
0   2010           98                         17                  0.173469
1   2011          160                         48                  0.300000
2   2012          120                         82                  0.683333
3   2013           98                         56                  0.571429
4   2014           90                         37                  0.411111
5   2015          116                         34                  0.293103
6   2016           91                         17                  0.186813
7   2017           70                          8                  0.114286
8   2018          100                         2

In [4]:
# ============================================================
# Mature-sample sensitivity for outcome extension
# N4 only: XGBoost + Winsor, fixed hyperparameters
# Sample: 2010-2021
# CV: 2010-2014->2015, 2010-2015->2016, 2010-2016->2017, 2010-2017->2018
# Final holdout: 2019-2021
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# -----------------------------
# Safety checks
# -----------------------------
assert TARGET_COL == "outcome_dissident_favorable", f"Unexpected TARGET_COL: {TARGET_COL}"
assert "year" in df.columns
assert "deal_number" in df.columns
assert TARGET_COL in df.columns
assert all(col in df.columns for col in all_predictors), "Some predictors are missing from df."

print("Running mature-sample sensitivity on N4 only.")
print("TARGET_COL:", TARGET_COL)
print("Total rows in full outcome panel:", len(df))

# -----------------------------
# Restrict to mature sample
# -----------------------------
mature_df = df[df["year"].between(2010, 2021)].copy()

print("\nRows in mature sample (2010-2021):", len(mature_df))
print("Years in mature sample:", sorted(mature_df["year"].unique()))

# Year-by-year summary
year_summary = (
    mature_df.groupby("year")[TARGET_COL]
    .agg(
        n_campaigns="size",
        dissident_favorable_count="sum",
        dissident_favorable_rate="mean"
    )
    .reset_index()
)
print("\nYear summary:")
print(year_summary)

# -----------------------------
# Sensitivity split design
# -----------------------------
fold_definitions = [
    {"fold": "fold_1", "train_years": list(range(2010, 2015)), "val_years": [2015]},
    {"fold": "fold_2", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_3", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_4", "train_years": list(range(2010, 2018)), "val_years": [2018]},
]

final_train_years = list(range(2010, 2019))
final_holdout_years = [2019, 2020, 2021]

split_summary_rows = []
for spec in fold_definitions:
    train_df_tmp = mature_df[mature_df["year"].isin(spec["train_years"])].copy()
    val_df_tmp = mature_df[mature_df["year"].isin(spec["val_years"])].copy()

    split_summary_rows.append({
        "split_name": spec["fold"],
        "train_years": f"{min(spec['train_years'])}-{max(spec['train_years'])}",
        "validation_years": ",".join(map(str, spec["val_years"])),
        "n_train": len(train_df_tmp),
        "train_positive_rate": train_df_tmp[TARGET_COL].mean(),
        "n_validation": len(val_df_tmp),
        "validation_positive_rate": val_df_tmp[TARGET_COL].mean(),
    })

final_train_df_tmp = mature_df[mature_df["year"].isin(final_train_years)].copy()
final_holdout_df_tmp = mature_df[mature_df["year"].isin(final_holdout_years)].copy()
split_summary_rows.append({
    "split_name": "final_holdout",
    "train_years": f"{min(final_train_years)}-{max(final_train_years)}",
    "validation_years": ",".join(map(str, final_holdout_years)),
    "n_train": len(final_train_df_tmp),
    "train_positive_rate": final_train_df_tmp[TARGET_COL].mean(),
    "n_validation": len(final_holdout_df_tmp),
    "validation_positive_rate": final_holdout_df_tmp[TARGET_COL].mean(),
})

split_summary_df = pd.DataFrame(split_summary_rows)
print("\nSplit summary:")
print(split_summary_df)

# -----------------------------
# Fixed N4 hyperparameters
# -----------------------------
xgb_params_n4 = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.03,
    "min_child_weight": 1,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

# -----------------------------
# Winsorization settings
# -----------------------------
WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99

# If you already defined a more precise winsor-eligible list in your notebook,
# replace this fallback with that exact list for full consistency.
winsor_eligible_features = continuous_features.copy()

# -----------------------------
# Helper functions
# -----------------------------
def build_n4_pipeline():
    """
    Build a fresh pipeline for each fold/final fit.
    Matches the N4 winsorized preprocessing philosophy.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("continuous", SimpleImputer(strategy="median"), continuous_features),
            ("binary", SimpleImputer(strategy="constant", fill_value=0), binary_features),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                [categorical_feature],
            ),
        ],
        remainder="drop",
    )

    model = XGBClassifier(**xgb_params_n4)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    return pipeline

def get_positive_class_probability(fitted_pipeline, X, positive_class=1):
    model = fitted_pipeline.named_steps["model"]
    class_list = list(model.classes_)
    if positive_class not in class_list:
        raise ValueError(f"Positive class {positive_class} not found in model.classes_: {class_list}")
    positive_index = class_list.index(positive_class)
    return fitted_pipeline.predict_proba(X)[:, positive_index]

def safe_roc_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def compute_metric_row(model_id, evaluation_stage, fold_name, y_true, y_prob, n_obs):
    return {
        "model_id": model_id,
        "evaluation_stage": evaluation_stage,
        "fold": fold_name,
        "n_obs": int(n_obs),
        "positive_rate": float(np.mean(y_true)),
        "mean_predicted_probability": float(np.mean(y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "roc_auc": float(safe_roc_auc(y_true, y_prob)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
    }

def fit_train_winsor_bounds(train_df, feature_list, lower_q=0.01, upper_q=0.99):
    bounds = {}
    for feature in feature_list:
        s = train_df[feature].dropna()
        if len(s) == 0:
            bounds[feature] = (np.nan, np.nan)
        else:
            bounds[feature] = (s.quantile(lower_q), s.quantile(upper_q))
    return bounds

def apply_winsor_bounds(df_in, bounds):
    df_out = df_in.copy()
    for feature, (lb, ub) in bounds.items():
        if feature in df_out.columns and pd.notna(lb) and pd.notna(ub):
            df_out[feature] = df_out[feature].clip(lower=lb, upper=ub)
    return df_out

# -----------------------------
# Run fold-by-fold CV
# -----------------------------
MODEL_ID = "N4_MatureSample_2010_2021_Holdout2019_2021"
PRED_COL = "pred_outcome_xgb_winsor_mature"

fold_metric_rows = []
prediction_rows = []

print("\nStarting mature-sample sensitivity CV for N4...")
print(f"Fixed hyperparameters: {xgb_params_n4}")

for spec in fold_definitions:
    fold_name = spec["fold"]
    train_df = mature_df[mature_df["year"].isin(spec["train_years"])].copy()
    val_df = mature_df[mature_df["year"].isin(spec["val_years"])].copy()

    # Train-only winsorization bounds
    winsor_bounds = fit_train_winsor_bounds(
        train_df=train_df,
        feature_list=winsor_eligible_features,
        lower_q=WINSOR_LOWER,
        upper_q=WINSOR_UPPER,
    )

    train_df_w = apply_winsor_bounds(train_df, winsor_bounds)
    val_df_w = apply_winsor_bounds(val_df, winsor_bounds)

    X_train = train_df_w[all_predictors].copy()
    y_train = train_df_w[TARGET_COL].copy()

    X_val = val_df_w[all_predictors].copy()
    y_val = val_df_w[TARGET_COL].copy()

    pipeline = build_n4_pipeline()
    pipeline.fit(X_train, y_train)

    model = pipeline.named_steps["model"]
    print(f"\n[{fold_name}] model.classes_ = {model.classes_}")
    assert list(model.classes_) == [0, 1], f"Unexpected class ordering: {model.classes_}"

    val_prob = get_positive_class_probability(pipeline, X_val, positive_class=1)

    mean_prob_y1 = float(np.mean(val_prob[y_val.to_numpy() == 1])) if (y_val == 1).any() else np.nan
    mean_prob_y0 = float(np.mean(val_prob[y_val.to_numpy() == 0])) if (y_val == 0).any() else np.nan

    print(f"[{fold_name}] train years: {min(spec['train_years'])}-{max(spec['train_years'])}")
    print(f"[{fold_name}] validation years: {spec['val_years']}")
    print(f"[{fold_name}] n_train = {len(train_df)}, train positive rate = {y_train.mean():.4f}")
    print(f"[{fold_name}] n_val = {len(val_df)}, val positive rate = {y_val.mean():.4f}")
    print(f"[{fold_name}] mean prob | y=1 = {mean_prob_y1:.6f}")
    print(f"[{fold_name}] mean prob | y=0 = {mean_prob_y0:.6f}")

    fold_metric_rows.append(
        compute_metric_row(
            model_id=MODEL_ID,
            evaluation_stage="dev_cv",
            fold_name=fold_name,
            y_true=y_val,
            y_prob=val_prob,
            n_obs=len(val_df),
        )
    )

    pred_df = val_df[[
        "deal_number", "year", "outcome_mgmt_favorable",
        "outcome_dissident_favorable", "outcome_class"
    ]].copy()
    pred_df["model_id"] = MODEL_ID
    pred_df["evaluation_stage"] = "dev_cv"
    pred_df["fold"] = fold_name
    pred_df[PRED_COL] = val_prob
    prediction_rows.append(pred_df)

fold_metrics_df = pd.DataFrame(fold_metric_rows)
print("\nFold metrics:")
print(fold_metrics_df)

# -----------------------------
# Final mature holdout
# -----------------------------
print("\nRefitting N4 on 2010-2018 and evaluating on mature holdout 2019-2021...")

dev_df = mature_df[mature_df["year"].isin(final_train_years)].copy()
holdout_df = mature_df[mature_df["year"].isin(final_holdout_years)].copy()

# Train-only winsorization bounds for final fit
final_winsor_bounds = fit_train_winsor_bounds(
    train_df=dev_df,
    feature_list=winsor_eligible_features,
    lower_q=WINSOR_LOWER,
    upper_q=WINSOR_UPPER,
)

dev_df_w = apply_winsor_bounds(dev_df, final_winsor_bounds)
holdout_df_w = apply_winsor_bounds(holdout_df, final_winsor_bounds)

X_dev = dev_df_w[all_predictors].copy()
y_dev = dev_df_w[TARGET_COL].copy()

X_holdout = holdout_df_w[all_predictors].copy()
y_holdout = holdout_df_w[TARGET_COL].copy()

final_pipeline = build_n4_pipeline()
final_pipeline.fit(X_dev, y_dev)

final_model = final_pipeline.named_steps["model"]
print("[final_holdout] model.classes_ =", final_model.classes_)
assert list(final_model.classes_) == [0, 1], f"Unexpected class ordering: {final_model.classes_}"

holdout_prob = get_positive_class_probability(final_pipeline, X_holdout, positive_class=1)

holdout_mean_prob_y1 = float(np.mean(holdout_prob[y_holdout.to_numpy() == 1])) if (y_holdout == 1).any() else np.nan
holdout_mean_prob_y0 = float(np.mean(holdout_prob[y_holdout.to_numpy() == 0])) if (y_holdout == 0).any() else np.nan

print(f"[final_holdout] n_train = {len(dev_df)}, train positive rate = {y_dev.mean():.4f}")
print(f"[final_holdout] n_holdout = {len(holdout_df)}, holdout positive rate = {y_holdout.mean():.4f}")
print(f"[final_holdout] mean prob | y=1 = {holdout_mean_prob_y1:.6f}")
print(f"[final_holdout] mean prob | y=0 = {holdout_mean_prob_y0:.6f}")

final_stage_row = compute_metric_row(
    model_id=MODEL_ID,
    evaluation_stage="final_holdout",
    fold_name="2019_2021_holdout",
    y_true=y_holdout,
    y_prob=holdout_prob,
    n_obs=len(holdout_df),
)

holdout_pred_df = holdout_df[[
    "deal_number", "year", "outcome_mgmt_favorable",
    "outcome_dissident_favorable", "outcome_class"
]].copy()
holdout_pred_df["model_id"] = MODEL_ID
holdout_pred_df["evaluation_stage"] = "final_holdout"
holdout_pred_df["fold"] = "2019_2021_holdout"
holdout_pred_df[PRED_COL] = holdout_prob
prediction_rows.append(holdout_pred_df)

predictions_df = pd.concat(prediction_rows, axis=0, ignore_index=True)

# -----------------------------
# Stage summary
# -----------------------------
dev_cv_summary = {
    "model_id": MODEL_ID,
    "evaluation_stage": "dev_cv_mean",
    "fold": "all_folds",
    "n_obs": int(fold_metrics_df["n_obs"].sum()),
    "positive_rate": float(np.average(
        fold_metrics_df["positive_rate"],
        weights=fold_metrics_df["n_obs"]
    )),
    "mean_predicted_probability": float(np.average(
        fold_metrics_df["mean_predicted_probability"],
        weights=fold_metrics_df["n_obs"]
    )),
    "pr_auc": float(fold_metrics_df["pr_auc"].mean()),
    "roc_auc": float(fold_metrics_df["roc_auc"].mean()),
    "brier_score": float(fold_metrics_df["brier_score"].mean()),
}

stage_metrics_df = pd.DataFrame([dev_cv_summary, final_stage_row])

print("\nStage metrics:")
print(stage_metrics_df)

# -----------------------------
# Save outputs
# -----------------------------
output_prefix = "Outcome_Model_08_XGB_Winsor_v1_MatureSample_2010_2021_Holdout2019_2021"

fold_metrics_path = Path(f"{output_prefix}_Fold_Metrics.csv")
stage_metrics_path = Path(f"{output_prefix}_Stage_Metrics.csv")
predictions_path = Path(f"{output_prefix}_Predictions.csv")
year_summary_path = Path(f"{output_prefix}_Year_Summary.csv")
split_summary_path = Path(f"{output_prefix}_Split_Summary.csv")

fold_metrics_df.to_csv(fold_metrics_path, index=False)
stage_metrics_df.to_csv(stage_metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
year_summary.to_csv(year_summary_path, index=False)
split_summary_df.to_csv(split_summary_path, index=False)

print("\nSaved files:")
print(" -", fold_metrics_path)
print(" -", stage_metrics_path)
print(" -", predictions_path)
print(" -", year_summary_path)
print(" -", split_summary_path)

print("\nMature-sample sensitivity run complete.")

Running mature-sample sensitivity on N4 only.
TARGET_COL: outcome_dissident_favorable
Total rows in full outcome panel: 1401

Rows in mature sample (2010-2021): 1168
Years in mature sample: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

Year summary:
    year  n_campaigns  dissident_favorable_count  dissident_favorable_rate
0   2010           98                         17                  0.173469
1   2011          160                         48                  0.300000
2   2012          120                         82                  0.683333
3   2013           98                         56                  0.571429
4   2014           90                         37                  0.411111
5   2015          116                         34                  0.293103
6   2016           91                         17                  0.186813
7   2017           70                          8                  0.114286
8   2018          100                         2